In [4]:
!pip install -q ultralytics pycocotools

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import time
import subprocess
from pathlib import Path
import torch
from ultralytics import YOLO

import io
import shutil
from collections import defaultdict
from contextlib import redirect_stdout
from zipfile import ZipFile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import yaml

from PIL import Image, ImageDraw
from IPython.display import display, Markdown
from tqdm.auto import tqdm
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [6]:
ROOT = Path("/teamspace/studios/this_studio")
RCLONE = ROOT / "bin" / "rclone"
RCLONE_CONFIG = ROOT / "rclone.conf"

DATA = ROOT / "detection_prepared"
RUN = ROOT / "runs" / "detect" / "D0_yolo26n_640"
LAST = RUN / "weights" / "last.pt"

DRIVE_RUNS = "gdrive:WasteScope/runs/detect"

In [7]:
def backup_to_drive(trainer, force=False):
    if not force and (trainer.epoch + 1) % 5 != 0:
        return
    
    run = Path(trainer.save_dir)
    started = time.monotonic()

    try:
        subprocess.run(
            [
                str(RCLONE), "copy",
                str(run),
                f"{DRIVE_RUNS}/{run.name}",
                "--config", str(RCLONE_CONFIG),
                "--retries", "5",
                "--stats", "0",
            ],
            check=True, timeout=180
        )
    except (OSError, subprocess.SubprocessError) as error:
        print(
            f"WARNING: Drive backup failed: {error}", flush=True
        )
        return

    print(
        f"Drive backup complete: {run.name} "
        f"({time.monotonic() - started:.1f}s)",
        flush=True
    )

In [8]:
def resume_d0():
    RCLONE.chmod(0o755)
    subprocess.run([str(RCLONE), "version"], check=True)

    model = YOLO(str(LAST))
    model.add_callback("on_model_save", backup_to_drive)
    model.add_callback(
        "on_train_end",
        lambda trainer: backup_to_drive(trainer, force=True),
    )

    metrics = model.train(
        resume=True,
        data=str(DATA / "data.yaml"),
        epochs=100,
        patience=20,
        imgsz=640,
        batch=136,
        device=0,
        workers=8,
        optimizer="MuSGD",
        seed=42,
        deterministic=True,
        amp=True,
        cache="ram",
        val=True,
        plots=True,
        save=True,
        project=str(RUN.parent),
        name=RUN.name,
        exist_ok=False
    )

    print(f"Precision: {metrics.box.mp:.4f}")
    print(f"Recall: {metrics.box.mr:.4f}")
    print(f"mAP50: {metrics.box.map50:.4f}")
    print(f"mAP50–95: {metrics.box.map:.4f}")
    print("Best weights:", model.trainer.best)

    return model, metrics

In [9]:
RUN_TRAINING = False

if RUN_TRAINING:
    model, metrics = resume_d0()
else:
    print("Training skipped. Continue with the results-analysis cells.")

Training skipped. Continue with the results-analysis cells.


**Result analysis**

In [10]:
DRIVE_ROOT = Path("/content/drive/MyDrive/WasteScope")
ROOT = Path("/content/wastescope")

RUN_NAME = "D0_yolo26n_640"
DRIVE_RUN = DRIVE_ROOT / "runs" / "detect" / RUN_NAME

RUN = ROOT / "runs" / "detect" / RUN_NAME
DATA = ROOT / "detection_prepared"
VAL_ZIP = DRIVE_ROOT / "prepared" / "detection" / "val.zip"

ROOT.mkdir(parents=True, exist_ok=True)
shutil.copytree(DRIVE_RUN, RUN, dirs_exist_ok=True)

BEST = RUN / "weights" / "best.pt"
args = yaml.safe_load((RUN / "args.yaml").read_text())
IMGSZ = int(args["imgsz"])

In [11]:
history = pd.read_csv(RUN / "results.csv")
history.columns = history.columns.str.strip()
history = history.sort_values("epoch")

selection = "metrics/mAP50-95(B)"
best_row = history.loc[
    history[selection].eq(history[selection].max())
].iloc[-1]

csv_metrics = pd.DataFrame(
    [{
        "Precision": best_row["metrics/precision(B)"],
        "Recall": best_row["metrics/recall(B)"],
        "mAP50": best_row["metrics/mAP50(B)"],
        "mAP50–95": best_row["metrics/mAP50-95(B)"],
    }],
    index=[f"CSV best epoch: {int(best_row['epoch'])}"]
)

print("Completed epochs:", int(history["epoch"].max()))
display(csv_metrics.style.format("{:.4f}"))

Completed epochs: 84


,Precision,Recall,mAP50,mAP50–95
CSV best epoch: 64,0.9189,0.7119,0.7731,0.5889


In [12]:
for filename in (
    "results.png",
    "BoxPR_curve.png",
    "confusion_matrix_normalized.png",
):
    path = RUN / filename
    if path.is_file():
        display(Markdown(f"**{filename}**"))
        display(Image.open(path))

**results.png**

**BoxPR_curve.png**

**confusion_matrix_normalized.png**

In [13]:
VAL_IMAGES = DATA / "val" / "images"
VAL_LABELS = DATA / "val" / "labels"

already_prepared = (
    len(list(VAL_IMAGES.glob("*.png"))) == 6643
    and len(list(VAL_LABELS.glob("*.txt"))) == 6643
)

if not already_prepared:
    assert VAL_ZIP.is_file(), f"Missing: {VAL_ZIP}"
    DATA.mkdir(parents=True, exist_ok=True)

    local_zip = ROOT / "val.zip"

    with ZipFile(VAL_ZIP) as archive:
        required = VAL_ZIP.stat().st_size + sum(
            item.file_size for item in archive.infolist()
        )

    available = shutil.disk_usage(ROOT).free
    assert available > required + 2 * 1024**3, (
        f"Need about {required / 1024**3 + 2:.1f} GiB free; "
        f"available {available / 1024**3:.1f} GiB"
    )

    print("Copying val.zip from Drive...", flush=True)
    shutil.copyfile(VAL_ZIP, local_zip)

    with ZipFile(local_zip) as archive:
        for item in tqdm(archive.infolist(), desc="Extracting validation"):
            archive.extract(item, DATA)

    local_zip.unlink()

assert len(list(VAL_IMAGES.glob("*.png"))) == 6643
assert len(list(VAL_LABELS.glob("*.txt"))) == 6643

print("Validation ready: 6,643 images and labels")

Copying val.zip from Drive...


Extracting validation:   0%|          | 0/13288 [00:00<?, ?it/s]

Validation ready: 6,643 images and labels


**best.pt validation and predictions**

In [14]:
DEVICE = 0
model = YOLO(str(BEST))
names = {int(cid): name for cid, name in model.names.items()}
class_ids = sorted(names)

EVAL_YAML = ROOT / "validation.yaml"
EVAL_YAML.write_text(
    yaml.safe_dump({
        "path": str(DATA),
        "train": "train/images",
        "val": "val/images",
        "names": names
    }),
    encoding="utf-8"
)

264

In [15]:
evaluation = {}

def capture_validation(validator):
    evaluation["predictions"] = [
        dict(prediction) for prediction in validator.jdict
    ]
    evaluation["class_map"] = list(validator.class_map)

model.add_callback("on_val_end", capture_validation)

In [ ]:
try:
    validation = model.val(
        data=str(EVAL_YAML),
        split="val",
        imgsz=IMGSZ,
        device=DEVICE,
        batch=16,
        workers=2,
        cache=False,
        half=False,
        conf=0.001,
        iou=0.7,
        max_det=300,
        augment=False,
        plots=True,
        save_json=True,
        project=str(ROOT / "evaluation"),
        name="D0_val",
        exist_ok=True
    )
finally:
    model.callbacks["on_val_end"].remove(capture_validation)

predictions = evaluation["predictions"]
category_ids = [int(evaluation["class_map"][cid]) for cid in class_ids]

rechecked = pd.DataFrame(
    [{
        "Precision": validation.box.mp,
        "Recall": validation.box.mr,
        "mAP50": validation.box.map50,
        "mAP50–95": validation.box.map
    }],
    index=["best.pt validation"]
)

display(rechecked.style.format("{:.4f}"))

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.178 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 121 layers, 2,391,436 parameters, 0 gradients, 5.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 217.2±53.6 MB/s, size: 2562.4 KB)
val: Scanning /content/wastescope/detection_prepared/val/labels... 6643 images, 4243 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 6643/6643 90.7it/s 1:13<0.0ss
val: New cache created: /content/wastescope/detection_prepared/val/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 75% ━━━━━━━━━─── 315/416 1.8it/s 4:25<56.1s